# 03 Outliers: build the curated layer
Reads the `clean_*` tables from notebook 02 and writes `curated_*` tables. Decisions applied (see the README's Data decisions and the outlier report):
- **Decision 9 (B):** confirmed error values are set to missing with a flag, and the original value is kept in its own column.
- **Decision 10 (D):** 2018 financial figures (first mSCOA year) are flagged as not usable for financial features. 2018 audit opinions and expenditure stay.
- **Decision 11 (F):** genuine extreme values are kept as they are. Ratios and log scales come in the features step.

Layers: **raw** (as downloaded), then **clean** (fixed and flagged), then **curated** (outliers handled, ready for features). No rows are deleted at any layer.

The last cell runs reconciliation checks: every check must print PASS.

In [ ]:
from pyspark.sql import functions as F  # Spark's toolbox of column functions

Z_THRESHOLD = 3.5  # robust z-score cut-off for flagging a municipality's total expenditure (3.5 is the standard; change it here if you decide otherwise)
FIRST_RELIABLE_FIN_YEAR = 2019  # decision 10: financial figures before 2019 (the 2018 mSCOA transition year) are not used for features

# Decision 9: confirmed capture errors in irregular, fruitless and unauthorised expenditure (see the outlier report, Finding 1).
# Each is more than 100 times the municipality's whole annual budget.
UIFW_ERRORS = [("DC15", 2012, "irregular"),      # O R Tambo District: R183,969 million reported
               ("NW385", 2012, "unauthorised")]  # Ramotshere Moiloa: R38,917 million reported
ITEMS = ["irregular", "fruitless", "unauthorised"]  # the three expenditure types in the clean_uifw table

clean_uifw = spark.table("clean_uifw")  # open the cleaned expenditure table from notebook 02
clean_incexp = spark.table("clean_incexp")  # open the cleaned income and expenditure table
clean_panel = spark.table("clean_muni_year_panel")  # open the cleaned municipality-year panel
muni = spark.table("clean_municipalities").select("demarcation_code", "category")  # municipality codes and types (A, B, C)

## 1. Irregular, fruitless and unauthorised expenditure: set confirmed errors to missing

In [ ]:
cur_uifw = clean_uifw  # start from the clean table; we add columns, we don't remove rows
for item in ITEMS:  # go through each expenditure type
    is_listed = F.lit(False)  # start with "not a listed error"...
    for code, year, bad_item in UIFW_ERRORS:  # ...then check every listed error
        if bad_item == item:  # only errors for this expenditure type matter here
            is_listed = is_listed | ((F.col("demarcation_code") == code) & (F.col("year") == year))  # mark True if this row is that municipality and year
    is_error = is_listed | (F.col(f"{item}_exp") < 0)  # also treat any negative amount as an error (expenditure incurred can't be negative)
    cur_uifw = (cur_uifw
        .withColumn(f"{item}_exp_original", F.col(f"{item}_exp"))  # keep the original value in its own column, for traceability
        .withColumn(f"{item}_error", is_error)  # True where the value was judged an error
        .withColumn(f"{item}_exp", F.when(is_error, F.lit(None)).otherwise(F.col(f"{item}_exp"))))  # set errors to missing; keep everything else

# Recompute the total. If any part is missing, the total is missing too (we don't guess).
cur_uifw = cur_uifw.withColumn("uifw_total", F.col("irregular_exp") + F.col("fruitless_exp") + F.col("unauthorised_exp"))  # adding a missing value gives missing
cur_uifw.write.mode("overwrite").format("delta").saveAsTable("curated_uifw")  # save as a Delta table
display(cur_uifw.where("irregular_error OR fruitless_error OR unauthorised_error")  # show the rows that were changed
        .select("demarcation_code", "year", *[c for i in ITEMS for c in (f"{i}_exp_original", f"{i}_error")]))  # with their original values and flags

## 2. Financial statements: flag years whose totals are broken

In [ ]:
tot = (clean_incexp  # start from income and expenditure line items
    .where((F.col("amount_type") == "AUDA") & F.col("item_code").between("3100", "4300"))  # audited actual figures, expense items only (codes 3100 to 4300)
    .groupBy("demarcation_code", "year")  # one row per municipality-year
    .agg(F.sum("amount").alias("total_expenditure"))  # add the expense items into total expenditure
    .join(muni, "demarcation_code", "left"))  # attach the municipality type

pos = tot.where("total_expenditure > 0").withColumn("log_exp", F.log10("total_expenditure"))  # positive totals only, on a log scale (logs need positive numbers)
med = pos.groupBy("category").agg(F.expr("percentile(log_exp, 0.5)").alias("med"))  # the exact median log total for each municipality type
pos = pos.join(med, "category")  # attach each type's median to its rows
mad = pos.groupBy("category").agg(F.expr("percentile(abs(log_exp - med), 0.5)").alias("mad"))  # MAD: the median distance from that median
pos = pos.join(mad, "category").withColumn("robust_z", 0.6745 * (F.col("log_exp") - F.col("med")) / F.col("mad"))  # robust z-score for every row

fin = (tot.join(pos.select("demarcation_code", "year", "robust_z"), ["demarcation_code", "year"], "left")  # attach the z-score (empty for non-positive totals)
    .withColumn("fin_error_reason",  # write down why a year is flagged, or leave it empty
        F.when(F.col("total_expenditure") <= 0, "total expenditure zero or negative")  # impossible for a working municipality
         .when(F.abs("robust_z") > Z_THRESHOLD, F.concat(F.lit("robust z beyond "), F.lit(str(Z_THRESHOLD)))))  # far outside its peer group
    .withColumn("fin_error", F.col("fin_error_reason").isNotNull()))  # True where there's a reason
fin.write.mode("overwrite").format("delta").saveAsTable("curated_fin_quality")  # save as a Delta table
display(fin.where("fin_error").orderBy("year", "demarcation_code"))  # show every flagged municipality-year

## 3. The curated panel: one row per municipality-year, ready for features

In [ ]:
uifw_cols = [c for c in cur_uifw.columns if c not in ("demarcation_code", "year")]  # every expenditure column from the curated table
panel = (clean_panel.drop(*[c for c in clean_panel.columns if c in uifw_cols])  # take the clean panel minus its old expenditure columns...
    .join(cur_uifw, ["demarcation_code", "year"], "left")  # ...and attach the curated ones instead
    .join(fin.select("demarcation_code", "year", "total_expenditure", "robust_z", "fin_error", "fin_error_reason"),  # attach the financial quality flags
          ["demarcation_code", "year"], "left")
    # fin_usable: True only when financial figures exist, come from 2019 or later (decision 10) and aren't flagged as broken (decision 9)
    .withColumn("fin_usable", F.coalesce(F.col("has_audited_incexp"), F.lit(False))  # audited figures exist...
                              & (F.col("year") >= FIRST_RELIABLE_FIN_YEAR)  # ...from 2019 on...
                              & ~F.coalesce(F.col("fin_error"), F.lit(False))))  # ...and aren't flagged
panel.write.mode("overwrite").format("delta").saveAsTable("curated_muni_year_panel")  # save as a Delta table
print(f"curated_muni_year_panel: {panel.count():,} rows, {len(panel.columns)} columns")  # report its size

## 4. Outlier log: one row for every value changed or flagged, for the audit trail

In [ ]:
uifw_log = None  # will hold the expenditure part of the log
for item in ITEMS:  # go through each expenditure type
    part = (cur_uifw.where(F.col(f"{item}_error"))  # rows where this type was set to missing
        .select("demarcation_code", "year",
                F.lit("uifw").alias("source"),  # which table it came from
                F.lit(item).alias("field"),  # which column
                F.col(f"{item}_exp_original").alias("original_value"),  # the value we removed
                F.lit("decision 9: confirmed capture error or negative amount, set to missing").alias("action")))  # what we did and why
    uifw_log = part if uifw_log is None else uifw_log.unionByName(part)  # stack the parts into one table

fin_log = (fin.where("fin_error")  # financial years flagged as broken
    .select("demarcation_code", "year",
            F.lit("incexp").alias("source"),  # which table it came from
            F.lit("total_expenditure").alias("field"),  # which figure
            F.col("total_expenditure").alias("original_value"),  # the value (not changed, only flagged)
            F.concat(F.lit("decision 9: "), F.col("fin_error_reason"), F.lit("; financial figures excluded from features")).alias("action")))  # why

log = uifw_log.unionByName(fin_log).orderBy("source", "year", "demarcation_code")  # one combined log, sorted
log.write.mode("overwrite").format("delta").saveAsTable("curated_outlier_log")  # save as a Delta table
display(log)  # show it

## 5. Reconciliation checks

In [ ]:
def check(name, ok, detail=""):
    """Print PASS or FAIL for one check, with optional detail in brackets."""
    print(("PASS " if ok else "FAIL ") + name + (f"  ({detail})" if detail else ""))  # e.g. "PASS no rows lost"

p = spark.table("curated_muni_year_panel")  # open the curated panel
check("no rows lost or added", p.count() == clean_panel.count(), f"{p.count()} vs {clean_panel.count()}")  # check 1: same rows as the clean panel

n_err = spark.table("curated_uifw").select(sum(F.col(f"{i}_error").cast("int") for i in ITEMS).alias("n")).agg(F.sum("n")).first()[0]  # count values set to missing
check("3 expenditure values set to missing (2 capture errors + 1 negative)", n_err == 3, n_err)  # check 2: exactly the ones in the report

removed = spark.table("curated_uifw").select(  # add up the original values of everything we set to missing
    sum(F.when(F.col(f"{i}_error"), F.col(f"{i}_exp_original")).otherwise(0) for i in ITEMS).alias("r")).agg(F.sum("r")).first()[0]
before = clean_uifw.select(sum(F.col(f"{i}_exp") for i in ITEMS).alias("t")).agg(F.sum("t")).first()[0]  # total before, from the clean table
after = spark.table("curated_uifw").select(sum(F.coalesce(F.col(f"{i}_exp"), F.lit(0)) for i in ITEMS).alias("t")).agg(F.sum("t")).first()[0]  # total after, counting missing as 0
check("total before = total after + removed values", abs(before - (after + removed)) < 1, f"R{before:,.0f} = R{after:,.0f} + R{removed:,.0f}")  # check 3: every rand accounted for

flagged = spark.table("curated_fin_quality").where("fin_error AND year BETWEEN 2019 AND 2024").count()  # broken financial years in 2019-2024
check("6 financial years flagged in 2019-2024 at z = 3.5", flagged == 6 or Z_THRESHOLD != 3.5, flagged)  # check 4: matches the report (only checked at 3.5)

usable_2018 = p.where("year = 2018 AND fin_usable").count()  # 2018 rows marked usable
check("no 2018 financials marked usable (decision 10)", usable_2018 == 0, usable_2018)  # check 5: 2018 is excluded from features

kept_2018 = p.where("year = 2018").count()  # 2018 rows still in the panel
check("2018 rows kept in the panel", kept_2018 == clean_panel.where("year = 2018").count(), kept_2018)  # check 6: excluded from features, not deleted

log_rows = spark.table("curated_outlier_log").count()  # rows in the outlier log
fin_all = spark.table("curated_fin_quality").where("fin_error").count()  # all flagged financial years, any year
check("outlier log covers every change and flag", log_rows == n_err + fin_all, f"{log_rows} rows")  # check 7: nothing changed without a log entry